# 1. Inicialização

In [203]:
%pip install xgboost 
%pip install -U "huggingface_hub"
!hf auth login
%pip install tabpfn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
User is already logged in.
Note: you may need to restart the kernel to use updated packages.


In [204]:
import pandas as pd
import numpy as np   

from xgboost import XGBClassifier
from tabpfn import TabPFNClassifier

from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, matthews_corrcoef
from huggingface_hub import hf_hub_download

from scipy.stats import wilcoxon 

In [205]:
# Novos imports necessários 
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import matthews_corrcoef
import numpy as np

# Constantes do TabPFN calibrado 
CALIBRATION_METHOD    = 'isotonic'
CALIBRATION_CV_FOLDS  = 3
THRESHOLD_LOWER_BOUND = 0.20
THRESHOLD_UPPER_BOUND = 0.50
THRESHOLD_STEP        = 0.01


In [206]:
df1 = pd.read_csv("https://github.com/Nertonm/tabpfn-xgboost-fairness-comparison/raw/refs/heads/main/data/prefeito_ceara.csv", encoding='latin1', sep=';')
df1["eleicao"] = 2024

df2 = pd.read_csv("https://github.com/Nertonm/tabpfn-xgboost-fairness-comparison/raw/refs/heads/main/data/prefeito_ceara2.csv", encoding='utf-8', sep=';')
df2["eleicao"] = 2020

df3 = pd.read_csv("https://github.com/Nertonm/tabpfn-xgboost-fairness-comparison/raw/refs/heads/main/data/prefeito_ceara3.csv", encoding='utf-8', sep=';')
df3["eleicao"] = 2016

for d in [df1, df2, df3]:
    if "Região" in d.columns:
        d.drop("Região", axis=1, inplace=True)

df = pd.concat([df1, df2, df3], ignore_index=True)
print(df.shape, df["eleicao"].value_counts().to_dict())
df.to_csv("dados_completos_eleicoes.csv", index=False, encoding='utf-8-sig', sep=';')


(1541, 16) {2020: 565, 2016: 502, 2024: 474}


In [207]:
display(df)

,Código município,Cor/raça,Estado civil,Faixa etária,Gênero,Grau de instrução,Município,Nome candidato,Número candidato,Ocupação,Partido,Situação totalização,Votos válidos,Votos nominais,Data de carga,eleicao
0,14630,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,MAURITI,JOÃO PAULO FURTADO,13,Prefeito,PT,Eleito,28644,20593,2026-02-28 19:30:47,2024
1,13358,Parda,Divorciado(a),45 a 49 anos,Masculino,Ensino Médio completo,BAIXIO,LUCIO ALVES BARROSO,10,Empresário,REPUBLICANOS,Eleito,4916,2542,2026-02-28 19:30:47,2024
2,13897,Branca,Casado(a),55 a 59 anos,Masculino,Superior completo,FORTALEZA,EVANDRO SA BARRETO LEITAO,13,Deputado,PT,Eleito,1421428,716133,2026-02-28 19:30:47,2024
3,15997,Parda,Divorciado(a),60 a 64 anos,Feminino,Superior completo,PARAIPABA,JOANA D' ARC BATISTA CARVALHO,55,Aposentado (Exceto Servidor Público),PSD,Não Eleito,20496,0,2026-02-28 19:30:47,2024
4,14753,Branca,Casado(a),35 a 39 anos,Masculino,Superior completo,MORADA NOVA,MARCO ANTONIO DE ARAUJO BICA JUNIOR,13,Advogado,PT,Não Eleito,48016,23206,2026-02-28 19:30:47,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1536,15857,Parda,Casado(a),50 a 54 anos,Masculino,Superior completo,MARACANAÚ,JOSE FIRMO CAMURÇA NETO,22,Prefeito,PR,Eleito,112784,81315,2026-01-19 14:01:28,2016
1537,14915,Parda,Casado(a),35 a 39 anos,Masculino,Superior completo,ORÓS,SIMAO PEDRO ALVES PEQUENO,55,Prefeito,PSD,Eleito,14084,8055,2026-01-19 14:01:28,2016
1538,13080,Parda,Casado(a),75 a 79 anos,Masculino,Superior completo,TURURU,RAIMUNDO SERPA BARROSO,15,Vereador,PMDB,Não Eleito,11534,4998,2026-01-19 14:01:28,2016
1539,15350,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,RERIUTABA,CLERTON ASSIS FURTADO,13,Empresário,PT,Não Eleito,11406,2746,2026-01-19 14:01:28,2016


# 2. Tratamento dos dados

In [208]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1541 entries, 0 to 1540
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Código município      1541 non-null   int64 
 1   Cor/raça              1541 non-null   object
 2   Estado civil          1541 non-null   object
 3   Faixa etária          1541 non-null   object
 4   Gênero                1541 non-null   object
 5   Grau de instrução     1541 non-null   object
 6   Município             1541 non-null   object
 7   Nome candidato        1541 non-null   object
 8   Número candidato      1541 non-null   int64 
 9   Ocupação              1541 non-null   object
 10  Partido               1541 non-null   object
 11  Situação totalização  1541 non-null   object
 12  Votos válidos         1541 non-null   int64 
 13  Votos nominais        1541 non-null   int64 
 14  Data de carga         1541 non-null   object
 15  eleicao               1541 non-null   

In [209]:
df = df[df["Situação totalização"] != "Segundo turno"]

In [210]:
df = df.drop(["Nome candidato", "Data de carga", "Município"], axis=1)

In [211]:
CAT_COLS = ['Cor/raça','Estado civil','Faixa etária','Gênero',
            'Grau de instrução','Ocupação','Partido']

# Evita que o modelo vaze informações
def encode_fold(X_train, X_test, cat_cols):
    X_tr, X_te = X_train.copy(), X_test.copy()
    for col in cat_cols:
        cats = X_tr[col].astype('category').cat.categories
        cat_map = {v: i for i, v in enumerate(cats)}
        X_tr[col] = X_tr[col].map(cat_map).astype(int)
        X_te[col] = X_te[col].map(cat_map).fillna(-1).astype(int)
    return X_tr, X_te

df['Eleito'] = (df['Situação totalização'] == 'Eleito').astype(int)

display(df)

,Código município,Cor/raça,Estado civil,Faixa etária,Gênero,Grau de instrução,Número candidato,Ocupação,Partido,Situação totalização,Votos válidos,Votos nominais,eleicao,Eleito
0,14630,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,13,Prefeito,PT,Eleito,28644,20593,2024,1
1,13358,Parda,Divorciado(a),45 a 49 anos,Masculino,Ensino Médio completo,10,Empresário,REPUBLICANOS,Eleito,4916,2542,2024,1
2,13897,Branca,Casado(a),55 a 59 anos,Masculino,Superior completo,13,Deputado,PT,Eleito,1421428,716133,2024,1
3,15997,Parda,Divorciado(a),60 a 64 anos,Feminino,Superior completo,55,Aposentado (Exceto Servidor Público),PSD,Não Eleito,20496,0,2024,0
4,14753,Branca,Casado(a),35 a 39 anos,Masculino,Superior completo,13,Advogado,PT,Não Eleito,48016,23206,2024,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1536,15857,Parda,Casado(a),50 a 54 anos,Masculino,Superior completo,22,Prefeito,PR,Eleito,112784,81315,2016,1
1537,14915,Parda,Casado(a),35 a 39 anos,Masculino,Superior completo,55,Prefeito,PSD,Eleito,14084,8055,2016,1
1538,13080,Parda,Casado(a),75 a 79 anos,Masculino,Superior completo,15,Vereador,PMDB,Não Eleito,11534,4998,2016,0
1539,15350,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,13,Empresário,PT,Não Eleito,11406,2746,2016,0


In [212]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1529 entries, 0 to 1540
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Código município      1529 non-null   int64 
 1   Cor/raça              1529 non-null   object
 2   Estado civil          1529 non-null   object
 3   Faixa etária          1529 non-null   object
 4   Gênero                1529 non-null   object
 5   Grau de instrução     1529 non-null   object
 6   Número candidato      1529 non-null   int64 
 7   Ocupação              1529 non-null   object
 8   Partido               1529 non-null   object
 9   Situação totalização  1529 non-null   object
 10  Votos válidos         1529 non-null   int64 
 11  Votos nominais        1529 non-null   int64 
 12  eleicao               1529 non-null   int64 
 13  Eleito                1529 non-null   int64 
dtypes: int64(6), object(8)
memory usage: 179.2+ KB


In [213]:
# df = df.drop(["Situação totalização", "Cor/raça", "Estado civil", "Faixa etária", "Gênero", "Grau de instrução", "Ocupação", "Partido", "Votos válidos", "Votos nominais"], axis=1)

In [214]:
display(df)

,Código município,Cor/raça,Estado civil,Faixa etária,Gênero,Grau de instrução,Número candidato,Ocupação,Partido,Situação totalização,Votos válidos,Votos nominais,eleicao,Eleito
0,14630,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,13,Prefeito,PT,Eleito,28644,20593,2024,1
1,13358,Parda,Divorciado(a),45 a 49 anos,Masculino,Ensino Médio completo,10,Empresário,REPUBLICANOS,Eleito,4916,2542,2024,1
2,13897,Branca,Casado(a),55 a 59 anos,Masculino,Superior completo,13,Deputado,PT,Eleito,1421428,716133,2024,1
3,15997,Parda,Divorciado(a),60 a 64 anos,Feminino,Superior completo,55,Aposentado (Exceto Servidor Público),PSD,Não Eleito,20496,0,2024,0
4,14753,Branca,Casado(a),35 a 39 anos,Masculino,Superior completo,13,Advogado,PT,Não Eleito,48016,23206,2024,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1536,15857,Parda,Casado(a),50 a 54 anos,Masculino,Superior completo,22,Prefeito,PR,Eleito,112784,81315,2016,1
1537,14915,Parda,Casado(a),35 a 39 anos,Masculino,Superior completo,55,Prefeito,PSD,Eleito,14084,8055,2016,1
1538,13080,Parda,Casado(a),75 a 79 anos,Masculino,Superior completo,15,Vereador,PMDB,Não Eleito,11534,4998,2016,0
1539,15350,Parda,Casado(a),40 a 44 anos,Masculino,Superior completo,13,Empresário,PT,Não Eleito,11406,2746,2016,0


# 3. Treinamento e teste dos modelos

In [215]:
def find_optimal_threshold(
    true_labels: np.ndarray,
    predicted_probs: np.ndarray,
    lower: float = THRESHOLD_LOWER_BOUND,
    upper: float = THRESHOLD_UPPER_BOUND,
    step: float  = THRESHOLD_STEP,
) -> tuple[float, float]:
    """
    Busca o limiar de decisão que maximiza o MCC
    sobre as probabilidades fornecidas.

    Retorna (melhor_threshold, melhor_mcc).
    Restringe a busca ao intervalo [lower, upper] para evitar
    limiares degenerados que predizem sempre uma única classe.
    """
    candidate_thresholds = np.arange(lower, upper + step, step)

    best_threshold = lower
    best_mcc       = -1.0

    for threshold in candidate_thresholds:
        binary_predictions = (predicted_probs >= threshold).astype(int)

        # Ignora limiares que resultam em predição de classe única
        if len(np.unique(binary_predictions)) < 2:
            continue

        mcc = matthews_corrcoef(true_labels, binary_predictions)
        if mcc > best_mcc:
            best_mcc       = mcc
            best_threshold = threshold

    return round(float(best_threshold), 2), round(float(best_mcc), 4)


In [216]:
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import matthews_corrcoef
from xgboost import XGBClassifier
from tabpfn import TabPFNClassifier

# Configurações Globais para a Calibração
CALIBRATION_METHOD = 'isotonic'
CALIBRATION_CV_FOLDS = 3
RANDOM_SEED = 42

def find_optimal_threshold(true_labels: np.ndarray, predicted_probs: np.ndarray) -> tuple[float, float]:
    """
    Vila um conjunto de limiares (de 0.20 a 0.50) para encontrar aquele que
    maximiza o coeficiente MCC.
    """
    best_threshold = 0.50
    best_mcc = -1.0
    
    # Testa thresholds de 0.20 até 0.50 em incrementos de 0.01
    for threshold in np.arange(0.20, 0.51, 0.01):
        binary_predictions = (predicted_probs >= threshold).astype(int)
        current_mcc = matthews_corrcoef(true_labels, binary_predictions)
        
        if current_mcc > best_mcc:
            best_mcc = current_mcc
            best_threshold = threshold
            
    return best_threshold, best_mcc

# Inicialização de Variáveis
results_xgboost = []
results_tabpfn = []

stratification_labels = df['Gênero'].astype(str) + '_' + df['Eleito'].astype(str)

features_full = df[FEATURE_COLS].copy()
target_full = df['Eleito'].values

outer_cross_validator = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

for fold_number, (train_indices, test_indices) in enumerate(outer_cross_validator.split(features_full, stratification_labels), start=1):
    
    # 1. Separação Treino / Teste
    features_train_raw = features_full.iloc[train_indices]
    features_test_raw = features_full.iloc[test_indices]
    
    target_train = target_full[train_indices]
    target_test = target_full[test_indices]

    # 2. Encoding Categórico (Garante ausência de data leakage)
    features_train, features_test = encode_fold(
        features_train_raw, features_test_raw, CAT_COLS
    )

    # 3. Extração de Atributos Sensíveis do conjunto de teste bruto
    gender_test = features_test_raw['Gênero'].map({'Feminino': 0, 'Masculino': 1}).values
    race_test = features_test_raw['Cor/raça'].values

    print(f"\n{'='*60}")
    print(f"Fold {fold_number} | Treino: {len(target_train)} amostras | Teste: {len(target_test)} amostras")
    print(f"Mulheres no conjunto de teste: {(gender_test == 0).sum()}")
    print(f"{'-'*60}")

    # Treinamento e Avaliação: XGBoost
    class_weight_ratio = (target_train == 0).sum() / (target_train == 1).sum()

    xgboost_model = XGBClassifier(
        eval_metric='logloss',
        scale_pos_weight=class_weight_ratio,
        random_state=RANDOM_SEED,
    )
    
    xgboost_model.fit(features_train, target_train)
    predictions_xgb = xgboost_model.predict(features_test)
    mcc_xgb = matthews_corrcoef(target_test, predictions_xgb)

    results_xgboost.append({
        'fold': fold_number,
        'predictions': predictions_xgb,
        'target_test': target_test,
        'gender_test': gender_test,
        'race_test': race_test,
        'MCC': mcc_xgb,
    })
    
    print(f" XGBoost | MCC: {mcc_xgb:.4f} | Taxa pred. vitória: {predictions_xgb.mean():.3f}")

    # Treinamento e Avaliação: TabPFN (Calibrado + Otimização de Limiar)
    # Passo A: Criação do modelo base e do calibrador isotônico
    base_tabpfn = TabPFNClassifier(device='cpu', ignore_pretraining_limits=True)
    
    tabpfn_calibrated = CalibratedClassifierCV(
        estimator=base_tabpfn,
        method=CALIBRATION_METHOD,
        cv=CALIBRATION_CV_FOLDS,
    )
    
    tabpfn_calibrated.fit(features_train, target_train)

    # Passo B: Extração de Probabilidades do Treino para otimizar o limiar
    # Utilizamos o predict_proba direto do modelo calibrado para ganhar velocidade.
    # Como o CalibratedClassifierCV usa validação cruzada interna, essas probabilidades
    # já são parcialmente Out-Of-Fold (OOF).
    train_probabilities = tabpfn_calibrated.predict_proba(features_train)[:, 1]

    # Passo C: Busca do Limiar Ótimo (Restrito ao conjunto de Treino)
    optimal_threshold, threshold_train_mcc = find_optimal_threshold(
        true_labels=target_train,
        predicted_probs=train_probabilities,
    )

    # Passo D: Inferência Final (No conjunto de Teste)
    test_probabilities = tabpfn_calibrated.predict_proba(features_test)[:, 1]
    predictions_pfn = (test_probabilities >= optimal_threshold).astype(int)
    mcc_pfn = matthews_corrcoef(target_test, predictions_pfn)

    results_tabpfn.append({
        'fold': fold_number,
        'predictions': predictions_pfn,
        'target_test': target_test,
        'gender_test': gender_test,
        'race_test': race_test,
        'optimal_threshold': optimal_threshold,
        'threshold_mcc_train': threshold_train_mcc,
        'MCC': mcc_pfn,
    })

    print(f" TabPFN  | MCC: {mcc_pfn:.4f} | Limiar: {optimal_threshold:.2f} | MCC(treino): {threshold_train_mcc:.4f}")
    print(f"         | Taxa pred. vitória: {predictions_pfn.mean():.3f} | Prevalência real: {target_test.mean():.3f}")


Fold 1 | Treino: 1223 amostras | Teste: 306 amostras
Mulheres no conjunto de teste: 50
------------------------------------------------------------
 XGBoost | MCC: 0.0924 | Taxa pred. vitória: 0.418
 TabPFN  | MCC: 0.1724 | Limiar: 0.38 | MCC(treino): 0.5172
         | Taxa pred. vitória: 0.588 | Prevalência real: 0.369

Fold 2 | Treino: 1223 amostras | Teste: 306 amostras
Mulheres no conjunto de teste: 50
------------------------------------------------------------
 XGBoost | MCC: 0.1378 | Taxa pred. vitória: 0.356
 TabPFN  | MCC: 0.1759 | Limiar: 0.27 | MCC(treino): 0.2884
         | Taxa pred. vitória: 0.781 | Prevalência real: 0.369

Fold 3 | Treino: 1223 amostras | Teste: 306 amostras
Mulheres no conjunto de teste: 51
------------------------------------------------------------
 XGBoost | MCC: 0.1740 | Taxa pred. vitória: 0.346
 TabPFN  | MCC: 0.2137 | Limiar: 0.27 | MCC(treino): 0.3219
         | Taxa pred. vitória: 0.722 | Prevalência real: 0.366

Fold 4 | Treino: 1223 amostras

## 3.1. Divisão em teste e treino

## 3.2. XGBoost

## 3.3. TabPFN

# 4. Comparação entre os modelos

In [219]:
import numpy as np
from scipy.stats import wilcoxon
from typing import Dict, Tuple, List, Any

def calculate_fairness_bootstrap(
    actual_labels: np.ndarray, 
    predicted_labels: np.ndarray, 
    sensitive_attributes: np.ndarray, 
    privileged_class_value: int = 1, 
    n_bootstrap_iterations: int = 1000,
    random_seed_value: int = 42
) -> Dict[str, Tuple[float, Tuple[float, float]]]:
    """
    Calcula métricas de equidade (Fairness) utilizando amostragem Bootstrap.
    
    O método Bootstrap cria múltiplas amostras aleatórias (com reposição) a partir dos dados.
    Isso permite calcular não apenas a média final de cada métrica, mas também a sua 
    estabilidade através de um Intervalo de Confiança (IC) de 95%.
    """
    # Garantindo que as entradas sejam arrays do NumPy para permitir operações vetorizadas
    actual_labels = np.asarray(actual_labels)
    predicted_labels = np.asarray(predicted_labels)
    sensitive_attributes = np.asarray(sensitive_attributes)
    
    # Listas para armazenar os resultados calculados em cada uma das iterações
    bootstrap_disparate_impact_ratios = []
    bootstrap_equal_opportunity_diffs = []
    bootstrap_false_negative_rate_diffs = []
    
    # Gerador de números aleatórios inicializado com semente fixa para reprodutibilidade
    random_number_generator = np.random.default_rng(random_seed_value)
    total_samples = len(actual_labels)
    
    for iteration in range(n_bootstrap_iterations):
        # 1. Amostragem Bootstrap: Seleciona índices aleatórios com reposição
        # Simula uma "nova" base de dados do mesmo tamanho, permitindo dados repetidos
        bootstrap_indices = random_number_generator.integers(0, total_samples, size=total_samples)
        
        bootstrapped_actual_labels = actual_labels[bootstrap_indices]
        bootstrapped_predicted_labels = predicted_labels[bootstrap_indices]
        bootstrapped_sensitive_attributes = sensitive_attributes[bootstrap_indices]
        
        # 2. Criação de máscaras booleanas para separar os grupos de forma clara
        mask_privileged_group = (bootstrapped_sensitive_attributes == privileged_class_value)
        mask_unprivileged_group = (bootstrapped_sensitive_attributes != privileged_class_value)
        
        # 3. Cálculo do Disparate Impact Ratio (DIR)
        # O DIR compara a taxa bruta com que o modelo prevê a classe positiva (ex: Eleito) para cada grupo.
        positive_prediction_rate_privileged = bootstrapped_predicted_labels[mask_privileged_group].mean()
        positive_prediction_rate_unprivileged = bootstrapped_predicted_labels[mask_unprivileged_group].mean()
        
        # Só calculamos a razão se o grupo privilegiado tiver pelo menos uma previsão positiva (evita divisão por zero)
        if positive_prediction_rate_privileged > 0:
            disparate_impact_ratio = positive_prediction_rate_unprivileged / positive_prediction_rate_privileged
            bootstrap_disparate_impact_ratios.append(disparate_impact_ratio)
            
        # 4. Criação de máscaras focadas APENAS nos casos que REALMENTE foram positivos (os verdadeiros eleitos)
        # Isso é essencial para calcular métricas de erro baseadas em oportunidade
        mask_actual_positive_cases = (bootstrapped_actual_labels == 1)
        
        mask_privileged_actual_positives = mask_privileged_group & mask_actual_positive_cases
        mask_unprivileged_actual_positives = mask_unprivileged_group & mask_actual_positive_cases
        
        # 5. Cálculo das métricas EOD e FNR
        # A amostra atual precisa ter pelo menos um caso real positivo de cada grupo para as métricas fazerem sentido
        if mask_privileged_actual_positives.sum() > 0 and mask_unprivileged_actual_positives.sum() > 0:
            
            # TPR (True Positive Rate): Dos que realmente ganharam na vida real, quantos o modelo conseguiu identificar?
            true_positive_rate_privileged = bootstrapped_predicted_labels[mask_privileged_actual_positives].mean()
            true_positive_rate_unprivileged = bootstrapped_predicted_labels[mask_unprivileged_actual_positives].mean()
            
            # FNR (False Negative Rate): Dos que ganharam, quantos o modelo disse que iam perder? (1 - TPR)
            false_negative_rate_privileged = 1.0 - true_positive_rate_privileged
            false_negative_rate_unprivileged = 1.0 - true_positive_rate_unprivileged
            
            # EOD (Equal Opportunity Difference): Diferença na taxa de acerto direto (TPR) entre os grupos
            equal_opportunity_difference = true_positive_rate_unprivileged - true_positive_rate_privileged
            bootstrap_equal_opportunity_diffs.append(equal_opportunity_difference)
            
            # Diferença na taxa de falsos negativos entre os grupos
            false_negative_rate_difference = false_negative_rate_unprivileged - false_negative_rate_privileged
            bootstrap_false_negative_rate_diffs.append(false_negative_rate_difference)

    # 6. Função auxiliar embutida para consolidar as listas de métricas no formato padrão
    def compute_statistics_from_bootstrap_array(metric_values_list: list) -> Tuple[float, Tuple[float, float]]:
        """Calcula a média e o Intervalo de Confiança de 95% ignorando NaNs."""
        if not metric_values_list: 
            return np.nan, (np.nan, np.nan)
        
        mean_value = np.nanmean(metric_values_list)
        confidence_interval_lower = np.nanpercentile(metric_values_list, 2.5)  # Limite inferior do IC95%
        confidence_interval_upper = np.nanpercentile(metric_values_list, 97.5) # Limite superior do IC95%
        
        return mean_value, (confidence_interval_lower, confidence_interval_upper)

    # 7. Retorno do dicionário estruturado com as estatísticas finais
    return {
        'DIR': compute_statistics_from_bootstrap_array(bootstrap_disparate_impact_ratios),
        'EOD': compute_statistics_from_bootstrap_array(bootstrap_equal_opportunity_diffs),
        'FNR_diff': compute_statistics_from_bootstrap_array(bootstrap_false_negative_rate_diffs),
    }

# Agregação e Relatório de Resultados
def print_model_evaluation(model_name_label: str, cross_validation_results: List[Dict[str, Any]]):
    """
    Agrupa os dados brutos de todos os folds da validação cruzada, 
    calcula as métricas de equidade na base inteira e imprime um relatório estruturado.
    """
    # Empilhando as predições, rótulos e gêneros de todos os 5 ciclos (folds) em arrays únicos
    concatenated_actual_labels = np.concatenate([fold_data['y_te'] for fold_data in cross_validation_results])
    concatenated_predicted_labels = np.concatenate([fold_data['preds'] for fold_data in cross_validation_results])
    concatenated_gender_attributes = np.concatenate([fold_data['gender_te'] for fold_data in cross_validation_results])
    
    # Gerando as métricas com intervalo de confiança baseadas na junção total
    fairness_metrics_results = calculate_fairness_bootstrap(
        actual_labels=concatenated_actual_labels, 
        predicted_labels=concatenated_predicted_labels, 
        sensitive_attributes=concatenated_gender_attributes
    )
    
    # Extraindo as pontuações MCC (Matthews Correlation Coefficient) calculadas na etapa anterior
    mcc_scores_list = [fold_data['MCC'] for fold_data in cross_validation_results]
    
    # Impressão do cabeçalho e da performance média com Desvio Padrão
    print(f"\n── {model_name_label} ──")
    print(f"  MCC: {np.mean(mcc_scores_list):.4f} ± {np.std(mcc_scores_list):.4f}")
    
    # Desempacotando as tuplas para alinhar os valores de fairness perfeitamente na tela
    dir_mean, (dir_ic_lower, dir_ic_upper) = fairness_metrics_results['DIR']
    print(f"  DIR:     {dir_mean:.3f}  IC95%: [{dir_ic_lower:.3f}, {dir_ic_upper:.3f}]")
    
    eod_mean, (eod_ic_lower, eod_ic_upper) = fairness_metrics_results['EOD']
    print(f"  EOD:     {eod_mean:.3f}  IC95%: [{eod_ic_lower:.3f}, {eod_ic_upper:.3f}]")
    
    fnr_diff_mean, (fnr_diff_ic_lower, fnr_diff_ic_upper) = fairness_metrics_results['FNR_diff']
    print(f"  ΔFNRf-m: {fnr_diff_mean:.3f}  IC95%: [{fnr_diff_ic_lower:.3f}, {fnr_diff_ic_upper:.3f}]")


# Processando as listas de resultados contendo os dados do XGBoost e do TabPFN
for model_label, model_results_data in [('XGBoost', results_xgb), ('TabPFN', results_pfn)]:
    print_model_evaluation(model_label, model_results_data)

# Teste Estatístico pareado (Wilcoxon) para investigar se a diferença de performance (MCC) é significativa
mcc_scores_xgboost = [fold_data['MCC'] for fold_data in results_xgb]
mcc_scores_tabpfn = [fold_data['MCC'] for fold_data in results_pfn]

wilcoxon_statistic, p_value_wilcoxon = wilcoxon(mcc_scores_xgboost, mcc_scores_tabpfn, alternative='two-sided')
print(f"\nWilcoxon XGBoost vs TabPFN: stat={wilcoxon_statistic:.3f}, p={p_value_wilcoxon:.4f}")


── XGBoost ──
  MCC: 0.1680 ± 0.0619
  DIR:     1.066  IC95%: [0.890, 1.242]
  EOD:     -0.025  IC95%: [-0.136, 0.085]
  ΔFNRf-m: 0.025  IC95%: [-0.085, 0.136]

── TabPFN ──
  MCC: 0.2481 ± 0.0386
  DIR:     1.089  IC95%: [0.838, 1.408]
  EOD:     0.043  IC95%: [-0.060, 0.143]
  ΔFNRf-m: -0.043  IC95%: [-0.143, 0.060]

Wilcoxon XGBoost vs TabPFN: stat=0.000, p=0.0625


In [ ]:
print("\n" + "═" * 55)
print("RESUMO  Diagnóstico do Threshold Moving por Fold")
print("═" * 55)
print(f"{'Fold':<6} {'Threshold':<12} {'MCC Treino':<14} "
      f"{'MCC Teste':<12} {'Taxa Pred.':<12} {'Prevalência'}")
print("─" * 55)

for r in results_pfn:
    print(f"  {r['fold']:<5} "
          f"{r['optimal_threshold']:<12.2f} "
          f"{r['threshold_mcc_train']:<14.4f} "
          f"{r['MCC']:<12.4f} "
          f"{r['predictions'].mean():<12.3f} "
          f"{r['target_test'].mean():.3f}")

mean_threshold  = np.mean([r['optimal_threshold']      for r in results_pfn])
mean_mcc_test   = np.mean([r['MCC']                    for r in results_pfn])
mean_pred_rate  = np.mean([r['predictions'].mean()     for r in results_pfn])
mean_prevalence = np.mean([r['target_test'].mean()     for r in results_pfn])

print("─" * 55)
print(f"  Média  {mean_threshold:<12.2f} {'—':<14} "
      f"{mean_mcc_test:<12.4f} {mean_pred_rate:<12.3f} {mean_prevalence:.3f}")
print("\n  Se Taxa Pred. ≈ Prevalência, o threshold corrigiu o conservadorismo.")
print("  Se MCC Treino >> MCC Teste, o threshold pode estar overfit ao treino.")



═══════════════════════════════════════════════════════
RESUMO — Diagnóstico do Threshold Moving por Fold
═══════════════════════════════════════════════════════
Fold   Threshold    MCC Treino     MCC Teste    Taxa Pred.   Prevalência
───────────────────────────────────────────────────────


KeyError: 'optimal_threshold'

## 4.1. Matriz de confusão

### 4.1.1. XGBoost

In [ ]:
print(confusion_matrix(y_test, y_pred_xgboost))

[[137  56]
 [ 57  56]]


### 4.1.2. TabPFN

In [ ]:
print(confusion_matrix(y_test, y_pred_tabpfn))

[[172  21]
 [ 66  47]]


## 4.2. Matthews Correlation Coeficient (MCC)

### 4.2.1. XGBoost

In [ ]:
mcc_xgb = matthews_corrcoef(y_test, y_pred_xgboost)
print(mcc_xgb)

0.2058023177333497


### 4.2.2. TabPFN

In [ ]:
mcc_pfn = matthews_corrcoef(y_test, y_pred_tabpfn)
print(mcc_pfn)

0.35652035646362223


# 5. Análise de viés

In [ ]:
import numpy as np
from typing import Dict, Tuple, Any

# Função Central de Fairness com Bootstrap 
def calculate_fairness_bootstrap(
    actual_labels: np.ndarray,
    predicted_labels: np.ndarray,
    sensitive_attributes: np.ndarray,
    privileged_class_value: int = 1,
    n_bootstrap_iterations: int = 1000,
    confidence_interval: float = 0.95
) -> Dict[str, Any]:
    """
    Calcula as métricas DIR, EOD e FNR com Intervalos de Confiança (IC) via Bootstrap.
    
    Args:
        actual_labels: Rótulos verdadeiros da classe alvo (ex: 1 para Eleito, 0 para Não Eleito).
        predicted_labels: Predições geradas pelo modelo.
        sensitive_attributes: Atributo protegido utilizado para a análise (ex: Gênero).
        privileged_class_value: Valor que identifica o grupo privilegiado no atributo sensível.
        n_bootstrap_iterations: Quantidade de reamostragens para o cálculo do IC.
        confidence_interval: Nível de confiança desejado (padrão de 0.95 representa 95%).
    """
    # 1. Conversão de segurança para arrays do NumPy
    actual_labels = np.asarray(actual_labels)
    predicted_labels = np.asarray(predicted_labels)
    sensitive_attributes = np.asarray(sensitive_attributes)
    
    # Gerador aleatório instanciado de forma segura e reprodutível
    random_generator = np.random.default_rng(42)
    total_samples = len(actual_labels)

    # Armazenamento temporário para as métricas calculadas em cada iteração
    bootstrap_disparate_impact_ratios = []
    bootstrap_equal_opportunity_diffs = []
    bootstrap_false_negative_rate_diffs = []

    # 2. Loop de Reamostragem (Bootstrap)
    for _ in range(n_bootstrap_iterations):
        # Seleção de índices com reposição para criar a "nova" amostra
        bootstrap_indices = random_generator.integers(0, total_samples, size=total_samples)
        
        bootstrapped_actuals = actual_labels[bootstrap_indices]
        bootstrapped_predictions = predicted_labels[bootstrap_indices]
        bootstrapped_sensitive = sensitive_attributes[bootstrap_indices]

        # Separação através de máscaras lógicas
        mask_privileged = (bootstrapped_sensitive == privileged_class_value)
        mask_unprivileged = (bootstrapped_sensitive != privileged_class_value)

        
        # Cálculo: Disparate Impact Ratio (DIR)
        count_privileged = mask_privileged.sum()
        count_unprivileged = mask_unprivileged.sum()
        
        positive_prediction_rate_privileged = (
            bootstrapped_predictions[mask_privileged].mean() if count_privileged > 0 else np.nan
        )
        positive_prediction_rate_unprivileged = (
            bootstrapped_predictions[mask_unprivileged].mean() if count_unprivileged > 0 else np.nan
        )

        # Armazena apenas se a taxa do grupo privilegiado não for zero ou NaN
        if not (np.isnan(positive_prediction_rate_privileged) or positive_prediction_rate_privileged == 0):
            disparate_impact_ratio = positive_prediction_rate_unprivileged / positive_prediction_rate_privileged
            bootstrap_disparate_impact_ratios.append(disparate_impact_ratio)

        # Cálculo: EOD e Diferença de FNR
        mask_actual_positives = (bootstrapped_actuals == 1)
        
        mask_privileged_actual_positives = mask_privileged & mask_actual_positives
        mask_unprivileged_actual_positives = mask_unprivileged & mask_actual_positives

        true_positive_rate_privileged = (
            bootstrapped_predictions[mask_privileged_actual_positives].mean() 
            if mask_privileged_actual_positives.sum() > 0 else np.nan
        )
        true_positive_rate_unprivileged = (
            bootstrapped_predictions[mask_unprivileged_actual_positives].mean() 
            if mask_unprivileged_actual_positives.sum() > 0 else np.nan
        )

        if not (np.isnan(true_positive_rate_privileged) or np.isnan(true_positive_rate_unprivileged)):
            equal_opportunity_difference = true_positive_rate_unprivileged - true_positive_rate_privileged
            bootstrap_equal_opportunity_diffs.append(equal_opportunity_difference)
            
            # FNR (Taxa de Falsos Negativos) é o complemento do TPR (1 - TPR)
            false_negative_rate_difference = (1 - true_positive_rate_unprivileged) - (1 - true_positive_rate_privileged)
            bootstrap_false_negative_rate_diffs.append(false_negative_rate_difference)

    # 3. Tratamento Estatístico e Retorno
    significance_level = (1.0 - confidence_interval) / 2.0
    lower_percentile = significance_level * 100
    upper_percentile = (1.0 - significance_level) * 100

    def calculate_confidence_interval(metric_array: list) -> Tuple[float, float]:
        """Calcula o IC com base na lista de valores do bootstrap, ignorando NaNs."""
        if not metric_array:
            return (np.nan, np.nan)
        return (
            np.nanpercentile(metric_array, lower_percentile),
            np.nanpercentile(metric_array, upper_percentile)
        )

    # O dicionário retornado foi padronizado para facilitar a leitura por outras funções
    return {
        'DIR': np.nanmean(bootstrap_disparate_impact_ratios),
        'DIR_IC': calculate_confidence_interval(bootstrap_disparate_impact_ratios),
        
        'EOD': np.nanmean(bootstrap_equal_opportunity_diffs),
        'EOD_IC': calculate_confidence_interval(bootstrap_equal_opportunity_diffs),
        
        'FNR_diff': np.nanmean(bootstrap_false_negative_rate_diffs),
        'FNR_IC': calculate_confidence_interval(bootstrap_false_negative_rate_diffs),
        
        # Estas métricas são calculadas na base original global
        'n_priv': int((sensitive_attributes == privileged_class_value).sum()),
        'n_unpriv': int((sensitive_attributes != privileged_class_value).sum()),
    }

# Função de Impressão de Resultados 
def print_fairness_metrics(model_label: str, metrics: Dict[str, Any]) -> None:
    """Imprime os relatórios de fairness de forma limpa e estruturada."""
    
    print(f"\n── {model_label} ──")
    print(f"  DIR      : {metrics['DIR']:.3f}  IC95% [{metrics['DIR_IC'][0]:.3f}, {metrics['DIR_IC'][1]:.3f}]")
    print(f"  EOD      : {metrics['EOD']:.3f}  IC95% [{metrics['EOD_IC'][0]:.3f}, {metrics['EOD_IC'][1]:.3f}]")
    print(f"  ΔFNR(u-p): {metrics['FNR_diff']:.3f}  IC95% [{metrics['FNR_IC'][0]:.3f}, {metrics['FNR_IC'][1]:.3f}]")
    print(f"  Amostras : {metrics['n_priv']} Privilegiados | {metrics['n_unpriv']} Não Privilegiados")

## 5.1. Taxa de vitória real entre os gêneros

In [ ]:
# Agrega predições de todos os folds (gerados na seção 3)
# Pressupõe que results_xgb e results_pfn foram criados no loop do K-Fold
# Cada entrada tem: y_te, preds, gender_te, race_te

results_xgb = cross_validation_results_xgboost
results_pfn = cross_validation_results_tabpfn


all_yt_xgb   = np.concatenate([r['y_te']      for r in results_xgb])
all_yp_xgb   = np.concatenate([r['preds']     for r in results_xgb])
all_gen_xgb  = np.concatenate([r['gender_te'] for r in results_xgb])  # 0=Fem, 1=Masc
all_race_xgb = np.concatenate([r['race_te']   for r in results_xgb])  # texto original

all_yt_pfn   = np.concatenate([r['y_te']      for r in results_pfn])
all_yp_pfn   = np.concatenate([r['preds']     for r in results_pfn])
all_gen_pfn  = np.concatenate([r['gender_te'] for r in results_pfn])
all_race_pfn = np.concatenate([r['race_te']   for r in results_pfn])

print(f"Total amostras de teste (acumulado): {len(all_yt_xgb)}")
print(f"Mulheres: {(all_gen_xgb==0).sum()} | Homens: {(all_gen_xgb==1).sum()}")
print(f"Raça/cor:\n{pd.Series(all_race_xgb).value_counts()}")


Total amostras de teste (acumulado): 1529
Mulheres: 253 | Homens: 1276
Raça/cor:
Branca           792
Parda            690
Preta             31
Não Informado      7
Amarela            5
Indígena           4
Name: count, dtype: int64


## 5.2. Previsão por gênero

In [ ]:
print("5.2  DISPARATE IMPACT RATIO (gênero)")
print("  Privilegiado: Masculino (1) | Desprivilegiado: Feminino (0)")

for label, yt, yp, sg in [
    ('XGBoost', all_yt_xgb, all_yp_xgb, all_gen_xgb),
    ('TabPFN',  all_yt_pfn, all_yp_pfn, all_gen_pfn),
]:
    # Taxa bruta (sem bootstrap)  equivalente ao groupby original
    mask_m = sg == 1
    mask_f = sg == 0
    rate_m = yp[mask_m].mean()
    rate_f = yp[mask_f].mean()
    print(f"\n {label} ")
    print(f"  Taxa predita vitória  Masculino: {rate_m:.3f} | Feminino: {rate_f:.3f}")
    r = calculate_fairness_bootstrap(yt, yp, sg, privileged_class_value=1)
    print_fairness_metrics(label, r)
    # DIR < 0.8 indica disparidade significativa (regra dos 4/5)
    flag = "abaixo do limiar 0.8 (regra dos 4/5)" if r['DIR'] < 0.8 else "dentro do limiar"
    print(f"  {flag}")


5.2  DISPARATE IMPACT RATIO (gênero)
  Privilegiado: Masculino (1) | Desprivilegiado: Feminino (0)

 XGBoost 
  Taxa predita vitória  Masculino: 0.379 | Feminino: 0.403

── XGBoost ──
  DIR      : 1.066  IC95% [0.890, 1.242]
  EOD      : -0.025  IC95% [-0.136, 0.085]
  ΔFNR(u-p): 0.025  IC95% [-0.085, 0.136]
  Amostras : 1276 Privilegiados | 253 Não Privilegiados
  dentro do limiar

 TabPFN 
  Taxa predita vitória  Masculino: 0.190 | Feminino: 0.206

── TabPFN ──
  DIR      : 1.089  IC95% [0.838, 1.408]
  EOD      : 0.043  IC95% [-0.060, 0.143]
  ΔFNR(u-p): -0.043  IC95% [-0.143, 0.060]
  Amostras : 1276 Privilegiados | 253 Não Privilegiados
  dentro do limiar


### 5.2.1. XGBoost

### 5.2.2. TabPFN

## 5.3. Falsos negativos por gênero

In [ ]:
print("5.3 FALSE NEGATIVE RATE por gênero")
print("  FNR = taxa de candidatas eleitas preditas como não-eleitas")

for label, yt, yp, sg in [
    ('XGBoost', all_yt_xgb, all_yp_xgb, all_gen_xgb),
    ('TabPFN',  all_yt_pfn, all_yp_pfn, all_gen_pfn),
]:
    fnr_m = 1 - yp[(sg==1) & (yt==1)].mean()
    fnr_f = 1 - yp[(sg==0) & (yt==1)].mean()
    print(f"\n {label} ")
    print(f"  FNR Masculino : {fnr_m:.3f}  (n eleitos={((sg==1)&(yt==1)).sum()})")
    print(f"  FNR Feminino  : {fnr_f:.3f}  (n eleitas={((sg==0)&(yt==1)).sum()})")
    r = calculate_fairness_bootstrap(yt, yp, sg, privileged_class_value=1)

    print(f"  ΔFNR (Fem-Masc): {r['FNR_diff']:.3f}  IC95%[{r['FNR_IC'][0]:.3f}, {r['FNR_IC'][1]:.3f}]")
    print(f"  EOD            : {r['EOD']:.3f}  IC95%[{r['EOD_IC'][0]:.3f}, {r['EOD_IC'][1]:.3f}]")


5.3 FALSE NEGATIVE RATE por gênero
  FNR = taxa de candidatas eleitas preditas como não-eleitas

 XGBoost 
  FNR Masculino : 0.505  (n eleitos=467)
  FNR Feminino  : 0.531  (n eleitas=96)
  ΔFNR (Fem-Masc): 0.025  IC95%[-0.085, 0.136]
  EOD            : -0.025  IC95%[-0.136, 0.085]

 TabPFN 
  FNR Masculino : 0.687  (n eleitos=467)
  FNR Feminino  : 0.646  (n eleitas=96)
  ΔFNR (Fem-Masc): -0.043  IC95%[-0.143, 0.060]
  EOD            : 0.043  IC95%[-0.060, 0.143]


### 5.3.1. XGBoost

### 5.3.2. TabPFN

# 5.4 Análise por Raça/Cor

In [ ]:
print("5.4 ANÁLISE DE VIÉS POR RAÇA/COR")
print("  Privilegiado: Branca | Desprivilegiado: Parda, Preta, demais")

# Binariza: 1=Branca (privilegiado), 0=Não-Branca
race_bin_xgb = (all_race_xgb == 'Branca').astype(int)
race_bin_pfn = (all_race_pfn == 'Branca').astype(int)

for label, yt, yp, rb in [
    ('XGBoost', all_yt_xgb, all_yp_xgb, race_bin_xgb),
    ('TabPFN',  all_yt_pfn, all_yp_pfn, race_bin_pfn),
]:
    r = bootstrap_fairness(yt, yp, rb, privileged_class_value=1)
    print(f"\ {label} (Branca vs Não-Branca) ")
    print_fair(label, r)

# Distribuição por grupo para contextualizar
print("\n Distribuição no test set (acumulado) ")
race_counts = pd.Series(all_race_xgb).value_counts()
print(race_counts.to_string())
print("\n⚠️  Grupos com n < 30 (Preta, Indígena, Amarela) têm ICs muito amplos.")
print("   Reportar apenas Branca vs Parda individualmente; demais: descritivo.")

# Branca vs Parda (os dois grupos com n suficiente)
for label, yt, yp, race_arr in [
    ('XGBoost', all_yt_xgb, all_yp_xgb, all_race_xgb),
    ('TabPFN',  all_yt_pfn, all_yp_pfn, all_race_pfn),
]:
    mask = np.isin(race_arr, ['Branca', 'Parda'])
    r = bootstrap_fairness(yt[mask], yp[mask],
                           (race_arr[mask] == 'Branca').astype(int),
                           privileged_class_value=1)
    print(f"\n {label} (Branca vs Parda somente) ")
    print_fair(label, r)


5.4 ANÁLISE DE VIÉS POR RAÇA/COR
  Privilegiado: Branca | Desprivilegiado: Parda, Preta, demais


<>:13: SyntaxWarning: invalid escape sequence '\ '
<>:13: SyntaxWarning: invalid escape sequence '\ '
/tmp/ipykernel_2594067/1989072225.py:13: SyntaxWarning: invalid escape sequence '\ '
  print(f"\ {label} (Branca vs Não-Branca) ")


TypeError: bootstrap_fairness() got an unexpected keyword argument 'privileged_class_value'

In [ ]:
print("5.5  TABELA-RESUMO ")

rows = []
for label, yt, yp, sg, rb in [
    ('XGBoost', all_yt_xgb, all_yp_xgb, all_gen_xgb, race_bin_xgb),
    ('TabPFN',  all_yt_pfn, all_yp_pfn, all_gen_pfn, race_bin_pfn),
]:
    rg = bootstrap_fairness(yt, yp, sg, priv_val=1)  # gênero
    rr = bootstrap_fairness(yt, yp, rb, priv_val=1)  # raça

    rows.append({
        'Modelo': label,
        'DIR Gênero': f"{rg['DIR']:.3f} [{rg['DIR_IC'][0]:.3f},{rg['DIR_IC'][1]:.3f}]",
        'EOD Gênero': f"{rg['EOD']:.3f} [{rg['EOD_IC'][0]:.3f},{rg['EOD_IC'][1]:.3f}]",
        'ΔFNR Gênero': f"{rg['FNR_diff']:.3f} [{rg['FNR_IC'][0]:.3f},{rg['FNR_IC'][1]:.3f}]",
        'DIR Raça*': f"{rr['DIR']:.3f} [{rr['DIR_IC'][0]:.3f},{rr['DIR_IC'][1]:.3f}]",
        'EOD Raça*': f"{rr['EOD']:.3f} [{rr['EOD_IC'][0]:.3f},{rr['EOD_IC'][1]:.3f}]",
    })

display(pd.DataFrame(rows).set_index('Modelo'))
print("* Raça: Branca (priv.) vs Parda (unpriv.) grupos com n suficiente")
print("  ICs calculados via bootstrap (n=1000, α=0.05) sobre K-Fold acumulado")


5.5  TABELA-RESUMO 


TypeError: bootstrap_fairness() got an unexpected keyword argument 'priv_val'